In [1]:
!pip install -q "numpy<2.0"
!pip install -q --upgrade crewai litellm google-generativeai google-genai chromadb sentence-transformers crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 874.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 55.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
x

In [38]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

Enter Gemini API Key: ··········
API key set


In [11]:
!pip uninstall -y numpy
!pip install numpy --upgrade --force-reinstall

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 35.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.40.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [1]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

05:58:12 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
05:58:12 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [46]:
llm = "gemini/gemini-2.5-flash"

print(llm)

gemini/gemini-2.5-flash


In [68]:
from crewai.tools import tool
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer


client_db = PersistentClient(path="./chroma_db")

col = client_db.get_or_create_collection(
    name="placement_kb"
)

embed = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


@tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results["documents"][0]

    return "\n---\n".join(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [69]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

In [70]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [71]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [72]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [73]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

In [74]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

In [75]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

In [14]:
result = await crew.kickoff_async()

print("\n=== FINAL OUTPUT ===\n")

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 92cb996e-efc0-40b4-b3ae-bf314861e6e9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 7edce423-1c76-46ac-bf7e-4ec539daaaa4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'query': 'TCS Digital placement preparation details for CSE student'}                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: ...


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│  Task: Create JSON-style progress summary for S001.                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "name": "Ravi Kumar",                                                                                        │
│    "target_company": "TCS Digital",                                                                             │
│    "target_role_focus": "CSE Placement",                                                                        │
│    "progress_stage": "Mock Interview & Targeted Skill Development",                                             │
│    "overall_status_summary": "Ravi Kumar has been provided with a set of mock interview questions for TCS       │
│  Digital. A strong sample answer and detailed preparation tips were provided for a key algorithmic question     │
│  (Maximum Subarray Sum). Note: Specific general placement preparation details for TCS Digital for CSE students  │
│  were not found in the knowledge base.",                                                                        │
│    "preparation_modules": [                                                                                     │
│      {                                                                                                          │
│        "module_name": "Mock Interview Questions Set 1",                                                         │
│        "status": "Provided",                                                                                    │
│        "details": "A list of 10 mock interview questions covering technical, problem-solving, and behavioral    │
│  aspects relevant to a TCS Digital profile. Questions cover topics from DSA, OOP, Cloud, SQL, System Design,    │
│  and behavioral elements."                                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "module_name": "Algorithmic Problem Solving (Kadane's Algorithm)",                                       │
│        "status": "Strong Performance Demonstrated (Sample)",                                                    │
│        "questions_addressed": [                                                                                 │
│          {                                                                                                      │
│            "question_id": 3,                                                                                    │
│            "question_summary": "Given an array of integers, find the maximum sum of a contiguous sub-array.     │
│  Discuss time and space complexity.",                                                                           │
│            "performance_analysis": "An excellent sample answer was provided, demonstrating a deep               │
│  understanding and clear articulation of Kadane's Algorithm. The response included a detailed intuition,        │
│  step-by-step algorithm, a clear example walkthrough, a

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create JSON-style progress summary for S001.                                                             │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===

```json
{
  "student_id": "S001",
  "name": "Ravi Kumar",
  "target_company": "TCS Digital",
  "target_role_focus": "CSE Placement",
  "progress_stage": "Mock Interview & Targeted Skill Development",
  "overall_status_summary": "Ravi Kumar has been provided with a set of mock interview questions for TCS Digital. A strong sample answer and detailed preparation tips were provided for a key algorithmic question (Maximum Subarray Sum). Note: Specific general placement preparation details for TCS Digital for CSE students were not found in the knowledge base.",
  "preparation_modules": [
    {
      "module_name": "Mock Interview Questions Set 1",
      "status": "Provided",
      "details": "A list of 10 mock interview questions covering technical, problem-solving, and behavioral aspects relevant to a TCS Digital profile. Questions cover topics from DSA, OOP, Cloud, SQL, System Design, and behavioral elements."
    },
    {
      "module_name": "Algorithmic Problem So

In [80]:
import os
import getpass

# remove old Gemini key
os.environ.pop("GEMINI_API_KEY", None)

# enter new Gemini key
os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [66]:
import os

print(os.environ["GEMINI_API_KEY"][:10])

AQ.Ab8RN6J


In [76]:
import asyncio

transcripts = []

for p in profiles[:1]:
    print(f"Running for {p['name']} → {p['target_company']}")

    result = await crew.kickoff_async()

    transcripts.append({
        "student": p["name"],
        "target": p["target_company"],
        "final_output": str(result)
    })

    await asyncio.sleep(30)   # optional pause between runs

Running for Ravi Kumar → TCS Digital


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 24ca1fda-b7f5-4f22-a9f2-52a31644ef7e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: f5359f24-ecab-40e6-b98f-c7d8ace0b1e5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#26) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'query': 'TCS Digital placement preparation for CSE students'}                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create JSON-style progress summary for S001.                                                             │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "student_name": "Ravi Kumar",                                                                                │
│    "target_company_role": "TCS Digital (CSE)",                                                                  │
│    "overall_summary": "Ravi Kumar demonstrates a strong foundational understanding of Object-Oriented           │
│  Programming principles, evidenced by a comprehensive and well-articulated sample answer for Question 3. His    │
│  ability to provide both descriptions and relatable real-world examples, along with explaining practical        │
│  benefits, is highly commendable. Continued focus on diverse examples and connecting concepts to maintainable   │
│  code practices will further enhance his readiness.",                                                           │
│    "mock_interview_questions": [                                                                                │
│      "Given an array of integers, describe an efficient algorithm to find the maximum sum of a contiguous       │
│  subarray. Can you explain your approach and its time complexity?",                                             │
│      "Explain the concept of threading versus multi-processing in an operating system. When would you choose    │
│  one over the other, and what are the advantages and disadvantages of each?",                                   │
│      "Describe the four pillars of Object-Oriented Programming (OOP) with real-world examples. How do these     │
│  principles contribute to writing better, more maintainable code?",                                             │
│      "Tell me about a significant academic or personal project you've worked on. What was your specific role,   │
│  what technical challenges did you face, and how did you overcome them?",                                       │
│      "Imagine you're designing a simple user authentication system for a web application. What are the key      │
│  security considerations you would keep in mind, and what technologies or practices would you employ to ensure  │
│  data protection and user privacy?",                                                                            │
│      "What specifically attracts you to TCS Digital, as opposed to other roles or companies? How do you see     │
│  yourself contributing to our digital transformation initiatives?",                                             │
│      "What do you consider your greatest strengths that would make you a valuable asset to TCS Digital? And     │
│  what is one area you're actively working to improve upon?",                                                    │
│      "Describe a situation where you had to quickly learn a new technology or framework for a project. What     │
│  was your approach to learning, and how did you collaborate with your team during this process?",               │
│      "The digital landscape is constantly evolving. What emerging technologies or trends are you most excited   │
│  about, and how do you keep yourself updated with these

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 24ca1fda-b7f5-4f22-a9f2-52a31644ef7e                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "student_name": "Ravi Kumar",                                                                                │
│    "target_company_role": "TCS Digital (CSE)",                                                                  │
│    "overall_summary": "Ravi Kumar demonstrates a strong foundational understanding of Object-Oriented           │
│  Programming principles, evidenced by a comprehensive and well-articulated sample answer for Question 3. His    │
│  ability to provide both descriptions and relatable real-world examples, along with explaining practical        │
│  benefits, is highly commendable. Continued focus on diverse examples and connecting concepts to maintainable   │
│  code practices will further enhance his readiness.",                                                           │
│    "mock_interview_questions": [                                                                                │
│      "Given an array of integers, describe an efficient algorithm to find the maximum sum of a contiguous       │
│  subarray. Can you explain your approach and its time complexity?",                                             │
│      "Explain the concept of threading versus multi-processing in an operating system. When would you choose    │
│  one over the other, and what are the advantages and disadvantages of each?",                                   │
│      "Describe the four pillars of Object-Oriented Programming (OOP) with real-world examples. How do these     │
│  principles contribute to writing better, more maintainable code?",                                             │
│      "Tell me about a significant academic or personal project you've worked on. What was your specific role,   │
│  what technical challenges did you face, and how did you overcome them?",                                       │
│      "Imagine you're designing a simple user authentication system for a web application. What are the key      │
│  security considerations you would keep in mind, and what technologies or practices would you employ to ensure  │
│  data protection and user privacy?",                                                                            │
│      "What specifically attracts you to TCS Digital, as opposed to other roles or companies? How do you see     │
│  yourself contributing to our digital transformation initiatives?",                                             │
│      "What do you consider your greatest strengths that would make you a valuable asset to TCS Digital? And     │
│  what is one area you're actively working to improve upon?",                                                    │
│      "Describe a situation where you had to quickly learn a new technology or framework for a project. What     │
│  was your approach to learning, and how did you collaborate with your team during this process?",               │
│      "The digital landscape is constantly evolving. What emerging technologies or trends are you most excited   │
│  about, and how do you keep yourself updated with thes

In [77]:
pathlib.Path(
    "day10_sprint5_transcripts.json"
).write_text(
    json.dumps(transcripts, indent=2)
)

print("Saved transcripts successfully")

Saved transcripts successfully


In [78]:
md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    md += t["final_output"]

    md += "\n\n---\n\n"

pathlib.Path(
    "day10_sprint5_report.md"
).write_text(md)

print("Markdown report created")

Markdown report created


In [79]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>